# WSI Feature Integration Pipeline

This notebook loads WSI feature arrays from `.npz` files, aligns them with patient exam metadata, checks overlap, and builds a final overlap table.

From there, gene expressions, TMB count and TMB values are added for each patient in this table. MYC and ccAB labels are also included. 

## 1) Load WSI feature files

In [14]:
import os
import numpy as np
import pandas as pd


def normalize_exam_name(x):
    name = str(x).strip()
    if name.endswith('.csv') or name.endswith('.npz'):
        name = name.rsplit('.', 1)[0]
    return name


# Directory containing the .npz files
# data_dir = "/app/data/yash-m2m/Data/Features/WSI Features"
data_dir = "/Users/yashpatel/Documents/Bocconi/Classes/Year 2/Semester 2/Thesis/Code/ccRCC-MultiModal-Analysis/Data/Features/WSI Features"

# Dictionary to store all arrays
arrays_dict = {}

# Get all .npz files
npz_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.npz')])

print(f"Found {len(npz_files)} .npz files")
print(f"First 5 files: {npz_files[:5]}")

Found 2907 .npz files
First 5 files: ['C3L-00004-21.npz', 'C3L-00004-26.npz', 'C3L-00010-21.npz', 'C3L-00010-26.npz', 'C3L-00011-21.npz']


In [15]:
# Load all arrays from .npz files
for npz_file in npz_files:
    file_path = os.path.join(data_dir, npz_file)
    file_name = os.path.splitext(npz_file)[0]  # Remove .npz extension
    
    # Load the .npz file
    data = np.load(file_path)
    
    # Store the array (each file has 'arr_0' key)
    arrays_dict[file_name] = data['arr_0']
    
    # Close the file to free memory
    data.close()

print(f"\nSuccessfully loaded {len(arrays_dict)} arrays")
print(f"Sample keys: {list(arrays_dict.keys())[:5]}")


Successfully loaded 2907 arrays
Sample keys: ['C3L-00004-21', 'C3L-00004-26', 'C3L-00010-21', 'C3L-00010-26', 'C3L-00011-21']


## 2) Quick sanity check on loaded arrays

In [16]:
# Access a specific array
specific_key = list(arrays_dict.keys())[0]
example_array = arrays_dict[specific_key]
print(f"Example - {specific_key} shape: {example_array.shape}")
example_array

Example - C3L-00004-21 shape: (1, 1024)


array([[0.07911438, 0.02529474, 0.00861926, ..., 0.00324976, 0.00616264,
        0.04557005]], dtype=float32)

## 3) Load patient metadata and normalize exam names

In [17]:
WSI_patientfiles = pd.read_csv("/Users/yashpatel/Documents/Bocconi/Classes/Year 2/Semester 2/Thesis/Code/ccRCC-MultiModal-Analysis/Data/Patient files/WSI_patientfiles.csv")

WSI_exams = [normalize_exam_name(x) for x in WSI_patientfiles["chosen_exam"]]

print(f"Total exam entries: {len(WSI_exams)}")
print(f"Unique normalized exams: {len(set(WSI_exams))}")
WSI_exams[:10]

Total exam entries: 618
Unique normalized exams: 618


['C3L-00004-26',
 'C3L-00010-26',
 'C3L-00011-26',
 'C3L-00026-21',
 'C3L-00079-26',
 'C3L-00088-22',
 'C3L-00096-26',
 'C3L-00097-26',
 'C3L-00103-26',
 'C3L-00165-26']

## 4) Compare metadata exams vs available feature arrays

In [18]:
# Compare normalized WSI exams with loaded .npz keys
wsi_exams_set = set(WSI_exams)

if len(arrays_dict) > 0:
    array_keys_set = set(arrays_dict.keys())
    source_label = "arrays_dict.keys()"
else:
    # Fallback: derive keys from .npz filenames in folder (without extension)
    if 'npz_files' in globals():
        array_keys_set = {os.path.splitext(f)[0] for f in npz_files}
    else:
        array_keys_set = {
            os.path.splitext(f)[0]
            for f in os.listdir(data_dir)
            if f.endswith('.npz')
        }
    source_label = "folder .npz filenames (arrays_dict currently empty)"

in_both = sorted(wsi_exams_set & array_keys_set)
only_in_wsi_exams = sorted(wsi_exams_set - array_keys_set)
only_in_arrays = sorted(array_keys_set - wsi_exams_set)

print(f"Comparison source for array keys: {source_label}")
print(f"In both: {len(in_both)}")
print(f"Only in WSI_exams: {len(only_in_wsi_exams)}")
print(f"Only in array-key side: {len(only_in_arrays)}")

print("\nSample (up to 20) only in WSI_exams:")
print(only_in_wsi_exams[:20])

print("\nSample (up to 20) only in array-key side:")
print(only_in_arrays[:20])

Comparison source for array keys: arrays_dict.keys()
In both: 618
Only in WSI_exams: 0
Only in array-key side: 2289

Sample (up to 20) only in WSI_exams:
[]

Sample (up to 20) only in array-key side:
['C3L-00004-21', 'C3L-00010-21', 'C3L-00011-21', 'C3L-00026-26', 'C3L-00079-21', 'C3L-00088-21', 'C3L-00088-26', 'C3L-00096-21', 'C3L-00097-21', 'C3L-00103-21', 'C3L-00103-22', 'C3L-00103-23', 'C3L-00165-21', 'C3L-00165-22', 'C3L-00165-23', 'C3L-00165-24', 'C3L-00165-25', 'C3L-00183-22', 'C3L-00183-23', 'C3L-00183-24']


## 5) Build final matched table (common cases)

In [19]:
# Build table for common cases: case_id, chosen_exam, and corresponding array from arrays_dict

def normalize_exam_name(x):
    name = str(x).strip()
    if name.endswith('.csv') or name.endswith('.npz'):
        name = name.rsplit('.', 1)[0]
    return name

if len(arrays_dict) == 0:
    raise ValueError("arrays_dict is empty. Run the array-loading cell first.")

# Add normalized key for matching
common_df = WSI_patientfiles.copy()
common_df["normalized_exam"] = common_df["chosen_exam"].apply(normalize_exam_name)

# Keep only rows present in both WSI_exams and arrays_dict keys
common_keys = set(common_df["normalized_exam"]) & set(arrays_dict.keys())
common_df = common_df[common_df["normalized_exam"].isin(common_keys)].copy()

# Attach the specific array from arrays_dict
common_df["array"] = common_df["normalized_exam"].map(arrays_dict)

# Final requested table
common_cases_table = common_df[["case_id", "chosen_exam", "array"]].reset_index(drop=True)

print(f"Rows in common_cases_table: {len(common_cases_table)}")
common_cases_table.head()

Rows in common_cases_table: 618


,case_id,chosen_exam,array
0,C3L-00004,C3L-00004-26.npz,"[[0.09093224, 0.025526874, 0.009832709, 0.0465..."
1,C3L-00010,C3L-00010-26.npz,"[[0.08283094, 0.024116952, 0.009357204, 0.0411..."
2,C3L-00011,C3L-00011-26.npz,"[[0.08987766, 0.034415565, 0.010371152, 0.0317..."
3,C3L-00026,C3L-00026-21.npz,"[[0.093995355, 0.040574342, 0.008823442, 0.032..."
4,C3L-00079,C3L-00079-26.npz,"[[0.074407145, 0.03540087, 0.0067873383, 0.052..."


## 6) Adding further patient metadata

In [20]:
patient_metadata_split = pd.read_csv("/Users/yashpatel/Documents/Bocconi/Classes/Year 2/Semester 2/Thesis/Code/ccRCC-MultiModal-Analysis/Data/clinical+genomic_split.csv")

In [21]:
patient_metadata_split.head()

,case_id,gender,age_diag,grade,cancer_history,ajcc_path_tumor_pt,ajcc_path_nodes_pn,ajcc_clin_metastasis_cm,ajcc_path_metastasis_pm,ajcc_path_tumor_stage,vital_status_12,race_Asian,race_Black or African American,race_Hispanic or Latino,race_White,race_other,VHL_mutation,PBMR1_mutation,TTN_mutation,Split
0,C3L-00004,1,6.0,3,NaN,7,0,1,0,3,1,0,0,0,1,0,1,1,0,train
1,C3L-00010,1,1.0,3,NaN,3,1,1,0,1,1,0,0,0,1,0,1,0,0,train
2,C3L-00011,0,5.0,4,1.0,8,0,2,2,3,0,0,0,0,1,0,1,0,1,train
3,C3L-00026,0,5.0,3,NaN,2,0,1,0,1,1,0,0,0,1,0,1,0,1,train
4,C3L-00079,1,3.0,3,NaN,8,2,1,0,3,0,0,0,0,1,0,1,0,0,train


In [22]:
# Merge common_cases_table with clinical/genomic metadata using case_id as key
common_cases_table_merge = common_cases_table.copy()
patient_metadata_split_merge = patient_metadata_split.copy()

common_cases_table_merge["case_id"] = common_cases_table_merge["case_id"].astype(str).str.strip()
patient_metadata_split_merge["case_id"] = patient_metadata_split_merge["case_id"].astype(str).str.strip()

merged_common_cases_table = common_cases_table_merge.merge(
    patient_metadata_split_merge,
    on="case_id",
    how="left",
    indicator=True,
    suffixes=("", "_clin")
)

print(f"Rows in common_cases_table: {len(common_cases_table_merge)}")
print(f"Rows after merge: {len(merged_common_cases_table)}")
print("Merge status counts:")
print(merged_common_cases_table["_merge"].value_counts())

# Keep final table without merge indicator
merged_common_cases_table = merged_common_cases_table.drop(columns=["_merge"])

# Set case_id as the dataframe index
merged_common_cases_table = merged_common_cases_table.set_index("case_id")

merged_common_cases_table.head()

Rows in common_cases_table: 618
Rows after merge: 618
Merge status counts:
_merge
both          618
left_only       0
right_only      0
Name: count, dtype: int64


,chosen_exam,array,gender,age_diag,grade,cancer_history,ajcc_path_tumor_pt,ajcc_path_nodes_pn,ajcc_clin_metastasis_cm,ajcc_path_metastasis_pm,...,vital_status_12,race_Asian,race_Black or African American,race_Hispanic or Latino,race_White,race_other,VHL_mutation,PBMR1_mutation,TTN_mutation,Split
case_id,,,,,,,,,,,,,,,,,,,,,
C3L-00004,C3L-00004-26.npz,"[[0.09093224, 0.025526874, 0.009832709, 0.0465...",1,6.0,3,NaN,7,0,1,0,...,1,0,0,0,1,0,1,1,0,train
C3L-00010,C3L-00010-26.npz,"[[0.08283094, 0.024116952, 0.009357204, 0.0411...",1,1.0,3,NaN,3,1,1,0,...,1,0,0,0,1,0,1,0,0,train
C3L-00011,C3L-00011-26.npz,"[[0.08987766, 0.034415565, 0.010371152, 0.0317...",0,5.0,4,1.0,8,0,2,2,...,0,0,0,0,1,0,1,0,1,train
C3L-00026,C3L-00026-21.npz,"[[0.093995355, 0.040574342, 0.008823442, 0.032...",0,5.0,3,NaN,2,0,1,0,...,1,0,0,0,1,0,1,0,1,train
C3L-00079,C3L-00079-26.npz,"[[0.074407145, 0.03540087, 0.0067873383, 0.052...",1,3.0,3,NaN,8,2,1,0,...,0,0,0,0,1,0,1,0,0,train


## 7) Adding gene expressions, TMB_Count, TMB_value, ccAB and MYC data

In [23]:
ccAB_data = pd.read_csv("/Users/yashpatel/Documents/Bocconi/Classes/Year 2/Semester 2/Thesis/Code/ccRCC-MultiModal-Analysis/Data/Gene expressions/DNA_feat_TCGA_ccAB.csv", index_col="PATIENT_ID")
print(ccAB_data.shape)
ccAB_data.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/yashpatel/Documents/Bocconi/Classes/Year 2/Semester 2/Thesis/Code/ccRCC-MultiModal-Analysis/Data/Gene expressions/DNA_feat_TCGA_ccAB.csv'

In [ ]:
MYC_data = pd.read_csv("/Users/yashpatel/Documents/Bocconi/Classes/Year 2/Semester 2/Thesis/Code/ccRCC-MultiModal-Analysis/Data/Gene expressions/tcga_DNA_MYC.csv", index_col="PATIENT_ID")
print(MYC_data.shape)
MYC_data.head()

(383, 167)


,Func_MSR1,Func_MUC4,Func_PTEN,Func_VHL,Func_CERCAM,Func_PBRM1,Func_PLEKHA2,Func_SETD2,Func_CD300E,Func_ANKRD36,...,Func_VCX2,Func_MAX,Func_HIST1H2BM,Func_EME2,Func_CELA1,Func_FRG2B,Func_GPRC6A,TMB_Count,TMB_Value,class_labels_MYC_V2_median
PATIENT_ID,,,,,,,,,,,,,,,,,,,,,
TCGA.A3.3308,LOF,WT,LOF,LOF,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,78,0.000003,above_median
TCGA.A3.3311,WT,WT,WT,WT,WT,WT,LOF,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,57,0.000002,above_median
TCGA.A3.3316,WT,WT,WT,WT,WT,LOF,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,43,0.000002,below_median
TCGA.A3.3317,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,61,0.000002,above_median
TCGA.A3.3320,WT,WT,WT,WT,WT,LOF,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,75,0.000003,below_median


In [ ]:
common_cols = ccAB_data.columns.intersection(MYC_data.columns).tolist()
common_patients = ccAB_data.index.intersection(MYC_data.index)

ccAB_sub = ccAB_data.loc[common_patients, common_cols]
MYC_sub = MYC_data.loc[common_patients, common_cols]

In [ ]:
print("Table contents are equal:", ccAB_sub.equals(MYC_sub))

if not ccAB_sub.equals(MYC_sub):
    try:
        pd.testing.assert_frame_equal(ccAB_sub, MYC_sub)
        print("frames are identical (assert passed)")
    except AssertionError as e:
        print("frames differ:", e)

Table contents are equal: True


In [ ]:
if ccAB_sub.equals(MYC_sub):
    gene_exp_data = ccAB_sub.copy()

    unique_ccAB_cols = [c for c in ccAB_data.columns if c not in common_cols]
    unique_MYC_cols = [c for c in MYC_data.columns if c not in common_cols]

    gene_exp_data = gene_exp_data.join(ccAB_data.loc[common_patients, unique_ccAB_cols])
    gene_exp_data = gene_exp_data.join(MYC_data.loc[common_patients, unique_MYC_cols])

    print("Gene expressions table shape:", gene_exp_data.shape)
else:
    raise ValueError("ccAB_sub and MYC_sub differ; cannot build a unified master table.")

gene_exp_data.index = gene_exp_data.index.str.replace(".", "-", regex=False)
gene_exp_data.head()

Gene expressions table shape: (383, 168)


,Func_MSR1,Func_MUC4,Func_PTEN,Func_VHL,Func_CERCAM,Func_PBRM1,Func_PLEKHA2,Func_SETD2,Func_CD300E,Func_ANKRD36,...,Func_MAX,Func_HIST1H2BM,Func_EME2,Func_CELA1,Func_FRG2B,Func_GPRC6A,TMB_Count,TMB_Value,class_labels,class_labels_MYC_V2_median
PATIENT_ID,,,,,,,,,,,,,,,,,,,,,
TCGA-A3-3308,LOF,WT,LOF,LOF,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,78,0.000003,ccB_only,above_median
TCGA-A3-3311,WT,WT,WT,WT,WT,WT,LOF,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,57,0.000002,NON_ccA/B,above_median
TCGA-A3-3316,WT,WT,WT,WT,WT,LOF,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,43,0.000002,ccB_only,below_median
TCGA-A3-3317,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,61,0.000002,ccB_only,above_median
TCGA-A3-3320,WT,WT,WT,WT,WT,LOF,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,75,0.000003,ccA_only,below_median


In [ ]:
# Find overlap between gene_exp_data patient IDs and merged_common_cases_table case IDs
gene_ids = set(gene_exp_data.index)
case_ids = set(merged_common_cases_table.index)

overlap_ids = gene_ids & case_ids

print("Overlap count:", len(overlap_ids))
print("Gene expression only (not in merged table):", len(gene_ids - case_ids))
print("Merged table only (not in gene expression):", len(case_ids - gene_ids))

Overlap count: 323
Gene expression only (not in merged table): 60
Merged table only (not in gene expression): 295


In [ ]:
# Build a full table for the overlapping IDs by joining the merged clinical/WSI table with gene expression data
full_data_table = merged_common_cases_table.join(gene_exp_data, how="inner")

print("Full table shape:", full_data_table.shape)
full_data_table.head()

Full table shape: (323, 189)


,chosen_exam,array,gender,age_diag,grade,cancer_history,ajcc_path_tumor_pt,ajcc_path_nodes_pn,ajcc_clin_metastasis_cm,ajcc_path_metastasis_pm,...,Func_MAX,Func_HIST1H2BM,Func_EME2,Func_CELA1,Func_FRG2B,Func_GPRC6A,TMB_Count,TMB_Value,class_labels,class_labels_MYC_V2_median
TCGA-A3-3311,TCGA-A3-3311-01A-01-BS1.npz,"[[0.09420455, 0.007828059, 0.0133514255, 0.020...",1,4.0,2,NaN,1,0,-1,1,...,WT,WT,WT,WT,WT,WT,57,0.000002,NON_ccA/B,above_median
TCGA-A3-3316,TCGA-A3-3316-11A-01-BS1.npz,"[[0.08240734, 0.0074756127, 0.011441833, 0.025...",1,4.0,3,NaN,4,0,-1,1,...,WT,WT,WT,WT,WT,WT,43,0.000002,ccB_only,below_median
TCGA-A3-3317,TCGA-A3-3317-01A-02-BS2.npz,"[[0.08040426, 0.016842486, 0.007083399, 0.0345...",1,5.0,2,NaN,4,1,-1,1,...,WT,WT,WT,WT,WT,WT,61,0.000002,ccB_only,above_median
TCGA-A3-3320,TCGA-A3-3320-01A-01-BS1.npz,"[[0.091176085, 0.0073955604, 0.0101240445, 0.0...",0,4.0,1,NaN,3,0,-1,1,...,WT,WT,WT,WT,WT,WT,75,0.000003,ccA_only,below_median
TCGA-A3-3322,TCGA-A3-3322-11A-01-TS1.npz,"[[0.07906659, 0.011373213, 0.0067032287, 0.027...",1,4.0,2,NaN,2,0,-1,1,...,WT,WT,WT,WT,WT,WT,48,0.000002,ccA_only,below_median


In [ ]:
# Save the full table to a CSV file
full_data_table.to_csv('/Users/yashpatel/Documents/Bocconi/Classes/Year 2/Semester 2/Thesis/Code/ccRCC-MultiModal-Analysis/Data/full_data_table.csv', index=True)

## Data Source Availability Summary

Create a comprehensive table showing which patients have data available from each of the three sources: WSI features, clinical metadata, and genomic data. Binary columns (0/1) indicate data availability for each patient. The genomic class labels are represented using one-hot encoding with binary columns: 'ccA', 'ccB', 'Non ccA/B', and 'shared ccA/B'. Additional genomic features ('TMB_Count', 'Func_PBRM1', 'Func_VHL', 'Func_BAP1') are also included. Patients without genomic data or missing values are represented as NaN.


In [ ]:
# Collect all unique patient IDs from all three sources
wsi_patients = set(common_cases_table['case_id'].unique())
metadata_patients = set(patient_metadata_split['case_id'].unique())
genomics_patients = set(gene_exp_data.index)

# Get all unique patients across all sources
all_patients = sorted(wsi_patients | metadata_patients | genomics_patients)

print(f"Patients with WSI data: {len(wsi_patients)}")
print(f"Patients with Metadata: {len(metadata_patients)}")
print(f"Patients with Genomics data: {len(genomics_patients)}")
print(f"Total unique patients: {len(all_patients)}")

# Create one-hot encoded columns for class labels and extract additional genomic features
# Initialize lists for each class type
ccA_list = []
ccB_list = []
non_ccAB_list = []
shared_ccAB_list = []

# Initialize lists for genomic features
tmb_count_list = []
func_pbrm1_list = []
func_vhl_list = []
func_bap1_list = []

for p in all_patients:
    if p in gene_exp_data.index and 'class_labels' in gene_exp_data.columns:
        label = gene_exp_data.loc[p, 'class_labels']
        # Map each label to the one-hot encoding
        if label == 'ccA_only':
            ccA_list.append(1)
            ccB_list.append(0)
            non_ccAB_list.append(0)
            shared_ccAB_list.append(0)
        elif label == 'ccB_only':
            ccA_list.append(0)
            ccB_list.append(1)
            non_ccAB_list.append(0)
            shared_ccAB_list.append(0)
        elif label == 'NON_ccA/B':
            ccA_list.append(0)
            ccB_list.append(0)
            non_ccAB_list.append(1)
            shared_ccAB_list.append(0)
        elif label == 'SHARED_ccA/B':
            ccA_list.append(0)
            ccB_list.append(0)
            non_ccAB_list.append(0)
            shared_ccAB_list.append(1)
        else:
            # Unexpected label, treat as NaN
            ccA_list.append(np.nan)
            ccB_list.append(np.nan)
            non_ccAB_list.append(np.nan)
            shared_ccAB_list.append(np.nan)
        
        # Extract genomic features
        tmb_count_list.append(gene_exp_data.loc[p, 'TMB_Count'] if 'TMB_Count' in gene_exp_data.columns else np.nan)
        func_pbrm1_list.append(gene_exp_data.loc[p, 'Func_PBRM1'] if 'Func_PBRM1' in gene_exp_data.columns else np.nan)
        func_vhl_list.append(gene_exp_data.loc[p, 'Func_VHL'] if 'Func_VHL' in gene_exp_data.columns else np.nan)
        func_bap1_list.append(gene_exp_data.loc[p, 'Func_BAP1'] if 'Func_BAP1' in gene_exp_data.columns else np.nan)
    else:
        # No genomics data, use NaN
        ccA_list.append(np.nan)
        ccB_list.append(np.nan)
        non_ccAB_list.append(np.nan)
        shared_ccAB_list.append(np.nan)
        tmb_count_list.append(np.nan)
        func_pbrm1_list.append(np.nan)
        func_vhl_list.append(np.nan)
        func_bap1_list.append(np.nan)

data_availability = pd.DataFrame({
    'PATIENT_ID': all_patients,
    'WSI': [1 if p in wsi_patients else 0 for p in all_patients],
    'Metadata': [1 if p in metadata_patients else 0 for p in all_patients],
    'Genomics': [1 if p in genomics_patients else 0 for p in all_patients],
    'ccA': ccA_list,
    'ccB': ccB_list,
    'Non ccA/B': non_ccAB_list,
    'shared ccA/B': shared_ccAB_list,
    'TMB_Count': tmb_count_list,
    'Func_PBRM1': func_pbrm1_list,
    'Func_VHL': func_vhl_list,
    'Func_BAP1': func_bap1_list
})

print("\nData availability summary:")
print(data_availability)

print("\nCombinations of data sources:")
print(data_availability.groupby(['WSI', 'Metadata', 'Genomics']).size().sort_index())

Patients with WSI data: 618
Patients with Metadata: 618
Patients with Genomics data: 383
Total unique patients: 678

Data availability summary:
       PATIENT_ID  WSI  Metadata  Genomics  ccA  ccB  Non ccA/B  shared ccA/B  \
0       C3L-00004    1         1         0  NaN  NaN        NaN           NaN   
1       C3L-00010    1         1         0  NaN  NaN        NaN           NaN   
2       C3L-00011    1         1         0  NaN  NaN        NaN           NaN   
3       C3L-00026    1         1         0  NaN  NaN        NaN           NaN   
4       C3L-00079    1         1         0  NaN  NaN        NaN           NaN   
..            ...  ...       ...       ...  ...  ...        ...           ...   
673  TCGA-DV-A4W0    1         1         0  NaN  NaN        NaN           NaN   
674  TCGA-EU-5904    1         1         0  NaN  NaN        NaN           NaN   
675  TCGA-G6-A5PC    1         1         0  NaN  NaN        NaN           NaN   
676  TCGA-G6-A8L7    1         1         0  Na

In [ ]:
# Save the data availability table to CSV
data_availability.to_csv('/Users/yashpatel/Documents/Bocconi/Classes/Year 2/Semester 2/Thesis/Code/ccRCC-MultiModal-Analysis/Data/data_availability_table.csv', index=False)
print("Data availability table saved to data_availability_table.csv")

# Display summary statistics
print("\nSummary Statistics:")
print(f"Patients with all 3 data sources: {len(data_availability[(data_availability['WSI'] == 1) & (data_availability['Metadata'] == 1) & (data_availability['Genomics'] == 1)])}")
print(f"Patients with WSI + Metadata: {len(data_availability[(data_availability['WSI'] == 1) & (data_availability['Metadata'] == 1)])}")
print(f"Patients with WSI + Genomics: {len(data_availability[(data_availability['WSI'] == 1) & (data_availability['Genomics'] == 1)])}")
print(f"Patients with Metadata + Genomics: {len(data_availability[(data_availability['Metadata'] == 1) & (data_availability['Genomics'] == 1)])}")


Data availability table saved to clinical+genomic_split.csv

Summary Statistics:
Patients with all 3 data sources: 323
Patients with WSI + Metadata: 618
Patients with WSI + Genomics: 323
Patients with Metadata + Genomics: 323
